# 01 · Raw data inventory: what we have, in what shape, and how we got it

**First notebook of the project.** Method: `docs/framework.md`. Findings per source go to `docs/source-cards/`.

The question the whole project answers: *After CUNY, where are the jobs in NYC, and are graduates ready for them?*

**What this notebook does:** reads `data/raw/` without changing anything and answers four things:
1. What files are on disk, and does each one have a provenance record (URL, date, SHA-256) in `data/raw/manifest.csv`?
2. **How** each source is obtained: API, bulk file download, archived copy, or scraped web page. This is the *Extract* step of the ETL.
3. **What format** each source arrives in (CSV, zipped CSV, Excel, tab-separated text, HTML).
4. A first look at every v1 source (brief O2): columns, a few rows, rough size.

**What it does not do:** no cleaning, no type decisions, no database. Those come per source in notebooks 02+.

Every read uses `dtype=str, keep_default_na=False`. The prototype lost work-experience values because pandas turned the text `"None"` into NaN; reading everything as raw text first stops that.

In [1]:
import sys, zipfile, gzip, io, re
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

# Works whether the kernel starts in notebooks/ (VS Code default) or the project root
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW, ROOT

pd.set_option("display.max_columns", 40, "display.max_colwidth", 60, "display.width", 200)
print("Raw folder:", RAW)

Raw folder: <repo>\data\raw


## 1. What's on disk

One row per file. `group/key` is the folder layout set by `src/sources.py`: `data/raw/<group>/<key>/<file>`. Files that don't follow that layout were placed by the prototype by hand.

In [2]:
def ext_of(p: Path) -> str:
    s = "".join(p.suffixes[-2:]).lower()
    return s if s in (".csv.gz",) else p.suffix.lower()

rows = []
for p in sorted(RAW.rglob("*")):
    if not p.is_file() or p.name in (".gitkeep", "manifest.csv"):
        continue
    rel = p.relative_to(RAW).as_posix()
    parts = rel.split("/")
    group, key = (parts[0], parts[1]) if len(parts) == 3 else (("prototype", parts[0]) if len(parts) == 2 else ("prototype", "(top level)"))
    rows.append({"file": rel, "group": group, "key": key, "ext": ext_of(p), "mb": p.stat().st_size / 1e6})
files = pd.DataFrame(rows)

print(f"{len(files)} files, {files.mb.sum()/1000:.2f} GB")
files.groupby("group").agg(files=("file", "size"), sources=("key", "nunique"), mb=("mb", "sum")).round(1).sort_values("mb", ascending=False)

253 files, 2.82 GB


,files,sources,mb
group,,,
jobs,26,11,1353.3
pay,13,2,641.9
hiring,5,2,288.7
outcomes,10,4,173.0
graduates,27,7,112.6
postings,6,3,95.9
bls_timeseries,25,3,65.3
prototype,98,2,61.1
skills,4,1,30.3


## 2. Provenance: does every file have a manifest record?

`manifest.csv` is the audit trail the download notebook (00) writes: source URL, download time (UTC), byte count and SHA-256 hash. A file without a record has no provable origin, so it can't feed v1 as-is.

In [3]:
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
print(f"manifest rows: {len(man)}  |  unique files: {man.file.nunique()}")

chk = files.merge(man[["file", "url", "downloaded_utc", "sha256", "note"]], on="file", how="outer", indicator=True)
chk["_merge"] = chk["_merge"].map({"both": "on disk + in manifest", "left_only": "on disk, NO manifest record",
                                   "right_only": "in manifest, MISSING on disk"})
display(chk["_merge"].value_counts().to_frame("files"))
chk[chk._merge != "on disk + in manifest"].groupby(["_merge", "key"]).agg(files=("file", "size"), mb=("mb", "sum")).round(1)

manifest rows: 155  |  unique files: 155


,files
_merge,
on disk + in manifest,155
"on disk, NO manifest record",98
"in manifest, MISSING on disk",0


files    mb
_merge                      key                     
on disk, NO manifest record (top level)     10  47.5
                            bls_matrix      88  13.6

In [4]:
# Size check: the manifest byte count should equal the file size on disk (catches truncated or replaced files)
both = chk[chk._merge == "on disk + in manifest"].merge(man[["file", "bytes"]], on="file")
both["disk_bytes"] = [ (RAW / f).stat().st_size for f in both.file ]
mismatch = both[both.bytes.astype(int) != both.disk_bytes]
print(f"{len(both)} manifested files; byte-count mismatches: {len(mismatch)}")
mismatch[["file", "bytes", "disk_bytes"]]

155 manifested files; byte-count mismatches: 0


,file,bytes,disk_bytes


The 98 files without a record are the prototype's: 10 files dropped at the top of `data/raw/` and 88 BLS web pages scraped into `bls_matrix/`. Are the top-level ones just copies of files the registry downloaded properly? Compare SHA-256 hashes.

In [5]:
import hashlib
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 22), b""): h.update(b)
    return h.hexdigest()

top = files[files.key == "(top level)"].copy()
top["sha256"] = top.file.map(lambda f: sha(RAW / f))
dup = top.merge(man[["file", "sha256"]].rename(columns={"file": "registry_copy"}), on="sha256", how="left")
dup[["file", "mb", "registry_copy"]].fillna({"registry_copy": "— no identical copy —"}).round(1)

,file,mb,registry_copy
0,C2024_A.zip,4.7,graduates/ipeds_completions/C2024_A.zip
1,C2024_A_Dict.zip,0.1,graduates/ipeds_dictionary/C2024_A_Dict.zip
2,CIP2020_SOC2018_Crosswalk.xlsx,0.4,graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx
3,education.xlsx,0.1,jobs/bls_ep_education/education_2026-10-12.xlsx
4,HD2024.zip,1.1,graduates/ipeds_directory/HD2024.zip
5,HD2024_Dict.zip,0.1,— no identical copy —
6,IES2020_CIP_SOC_Crosswalk_508C.pdf,0.2,— no identical copy —
7,Job Zones.txt,0.0,— no identical copy —
8,ny_dol_long_term_projections_pqm4-9qqb.csv,0.8,— no identical copy —
9,oesm25ma.zip,39.9,jobs/bls_oews_metro/oesm25ma.zip


## 3. How each source is obtained (the *Extract* step)

Classified from the download URL in the manifest:

| Method | What it means | Example host |
|---|---|---|
| **Socrata API: full export** | Open-data portal; `/api/views/<id>/rows.csv` returns the whole table as CSV. No key | data.ny.gov, data.cityofnewyork.us |
| **Socrata API: SoQL query** | Same portal, but `/resource/<id>.csv?$select=…&$where=…` filters on the server (used for NYC payroll, to leave out employee names) | data.cityofnewyork.us |
| **Bulk file download** | A published file (zip / xlsx / csv / gz) at a fixed URL | bls.gov, census.gov, nces.ed.gov, bea.gov, dol.gov, onetcenter.org |
| **Archived copy** | An older version recovered from the Wayback Machine because the publisher overwrote it | web.archive.org |
| **Web page (scrape later)** | An HTML page saved as-is; the table inside still has to be parsed | op.nysed.gov |
| **GitHub raw file** | A CSV published in a public repository | raw.githubusercontent.com |

In [6]:
def method(url: str) -> str:
    if not url:                      return "unknown (no manifest)"
    if "web.archive.org" in url:     return "Archived copy (Wayback)"
    if "/resource/" in url:          return "Socrata API: SoQL query"
    if "/api/views/" in url:         return "Socrata API: full export"
    if "githubusercontent" in url:   return "GitHub raw file"
    if re.search(r"\.(zip|xlsx|csv|gz|txt)(\?|$)", url) or "time.series" in url or url.endswith(("md5sum",)):
        return "Bulk file download"
    return "Web page (scrape later)"

chk["method"] = chk.url.fillna("").map(method)
chk["host"] = chk.url.fillna("").str.extract(r"https?://([^/]+)")[0].fillna("")
chk.groupby("method").agg(sources=("key", "nunique"), files=("file", "size"), mb=("mb", "sum")).round(1).sort_values("files", ascending=False)

,sources,files,mb
method,,,
unknown (no manifest),2,98,61.1
Bulk file download,20,92,1913.7
Web page (scrape later),3,38,1.8
Socrata API: full export,11,11,202.7
Socrata API: SoQL query,1,7,565.1
GitHub raw file,1,4,77.4
Archived copy (Wayback),1,3,2.0


## 4. File formats

Two levels: the format of the file as downloaded, and what sits *inside* the zips (which is what we actually parse).

In [7]:
display(files.groupby("ext").agg(files=("file", "size"), mb=("mb", "sum")).round(1).sort_values("files", ascending=False))

inside = []
for f in files[files.ext == ".zip"].file:
    with zipfile.ZipFile(RAW / f) as z:
        for i in z.infolist():
            if not i.is_dir():
                inside.append({"zip": f, "member": i.filename, "ext": Path(i.filename).suffix.lower(), "mb_unzipped": i.file_size / 1e6})
inside = pd.DataFrame(inside)
print("Inside the zips:")
inside.groupby("ext").agg(members=("member", "size"), mb_unzipped=("mb_unzipped", "sum")).round(1).sort_values("members", ascending=False)

,files,mb
ext,,
.html,126,15.4
.zip,46,734.4
.csv,30,849.8
.txt,28,67.0
.xlsx,17,856.0
.csv.gz,4,301.1
.md5sum,1,0.0
.pdf,1,0.2


Inside the zips:


,members,mb_unzipped
ext,,
.csv,218,2656.0
.xlsx,47,346.5
.txt,45,100.1
.pdf,7,1.5
.html,2,0.0
.xml,2,0.1


## 5. The v1 cut (brief O2) against what's on disk

brief §7 proposes which sources are in v1 and which wait. Tagging every source key that way shows what we actually have to explore before Oct 23. *Support* = not a headline source but needed to translate codes or tie out totals.

In [8]:
V1 = {  # key -> (role, the brief question it serves)
    "nysdol_projections":         ("v1", "Q1 jobs growing/shrinking + openings (current vintage 2022-32)"),
    "nysdol_projections_archive": ("v1", "Q1 older NYC vintages 2016-26, 2018-28, 2020-30"),
    "bls_oews_metro":             ("v1", "Q1/Q4 employment + pay by job over time, NY metro"),
    "nysdol_oews_regions":        ("v1", "Q4 NYC 5-borough wages incl. entry wage"),
    "nysdol_qcew_annual":         ("v1", "Q1 jobs by industry x borough over time"),
    "census_qwi":                 ("v1", "Q1/Q4 hiring + separations by industry x county x age"),
    "nysdol_ces":                 ("v1", "Q1 monthly jobs by industry"),
    "bea_county_gdp":             ("v1", "Q1 industry output by borough"),
    "ipeds_completions":          ("v1", "Q3 graduates by college x major x award x year"),
    "ipeds_directory":            ("v1", "Q3 which colleges are in NYC / CUNY"),
    "cuny_programs":              ("v1", "Q3 CUNY grads to 2024-25 + enrollment"),
    "bls_ep_education":           ("v1", "Q2 degree / experience / training each job needs"),
    "bls_ind_occ_matrix":         ("v1", "S8 industry -> job staffing pattern"),
    "acs_pums":                   ("v1", "Q2/Q4 what NYC grads actually do and earn"),
    "onet":                       ("v1", "Q2 skills per job"),
    "bls_cu":                     ("v1", "Q4 CPI for real (inflation-adjusted) pay"),
    "nces_cip":                   ("support", "CIP list, CIP 2010->2020, CIP->SOC crosswalk"),
    "bls_soc_crosswalk":          ("support", "SOC 2010->2018 for OEWS before May 2021"),
    "ipeds_dictionary":           ("support", "IPEDS column meanings"),
    "cuny_degrees_by_type":       ("support", "tie-out for CUNY totals"),
    "cuny_enrollment":            ("support", "tie-out for CUNY enrollment"),
}
LATER = {"nysed_registrants", "nysed_license_stats_index", "nysed_nclex", "nys_license_soc", "nyc_jobs", "nyc_civil_titles",
         "nyc_payroll", "indeed_hiring_lab", "census_pseo", "dol_oflc_wages", "nyfed_recent_grads", "scorecard_fos",
         "nysdol_ui_beneficiaries", "bls_jt"}

def role(row):
    if row.group == "prototype": return "prototype leftover"
    if row.key in V1:            return V1[row.key][0]
    return "later" if row.key in LATER else "not in brief cut"

chk["role"] = chk.apply(role, axis=1)
src = (chk.groupby(["role", "group", "key"])
          .agg(files=("file", "size"), mb=("mb", "sum"), method=("method", "first"), ext=("ext", lambda s: ", ".join(sorted(set(s)))))
          .round(1).reset_index())
src["serves"] = src.key.map(lambda k: V1.get(k, ("", ""))[1])
order = {"v1": 0, "support": 1, "later": 2, "not in brief cut": 3, "prototype leftover": 4}
src = src.sort_values(["role", "group", "key"], key=lambda s: s.map(order) if s.name == "role" else s).reset_index(drop=True)
display(src.groupby("role", sort=False).agg(sources=("key", "size"), files=("files", "sum"), mb=("mb", "sum")).round(0))
src

,sources,files,mb
role,,,
v1,16,60,1083.0
support,5,7,2.0
later,14,75,806.0
not in brief cut,2,13,872.0
prototype leftover,2,98,61.0


,role,group,key,files,mb,method,ext,serves
0,v1,bls_timeseries,bls_cu,7,4.1,Bulk file download,.txt,Q4 CPI for real (inflation-adjusted) pay
1,v1,graduates,cuny_programs,1,16.7,Socrata API: full export,.csv,Q3 CUNY grads to 2024-25 + enrollment
2,v1,graduates,ipeds_completions,10,82.8,Bulk file download,.zip,Q3 graduates by college x major x award x year
3,v1,graduates,ipeds_directory,10,10.9,Bulk file download,.zip,Q3 which colleges are in NYC / CUNY
4,v1,hiring,census_qwi,4,287.9,Bulk file download,".csv.gz, .md5sum, .txt",Q1/Q4 hiring + separations by industry x county x age
5,v1,jobs,bea_county_gdp,2,30.7,Bulk file download,.zip,Q1 industry output by borough
6,v1,jobs,bls_ep_education,1,0.1,Bulk file download,.xlsx,Q2 degree / experience / training each job needs
7,v1,jobs,bls_ind_occ_matrix,1,7.9,Bulk file download,.xlsx,S8 industry -> job staffing pattern
8,v1,jobs,bls_oews_metro,8,300.3,Bulk file download,.zip,"Q1/Q4 employment + pay by job over time, NY metro"
9,v1,jobs,nysdol_ces,1,37.1,Socrata API: full export,.csv,Q1 monthly jobs by industry


## 6. First look at each v1 source

For each v1 / support source: one representative file (the newest where there are several years), its columns, and the first rows. Sizes:
- **lines** is a raw newline count, so it's only *approximately* the row count (quoted text can hold line breaks). Exact row counts come in each source's own notebook.
- Big compressed files aren't counted here; we only read their header.

In [9]:
from IPython.display import display, Markdown

def count_lines(path: Path) -> int:
    n = 0
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            n += chunk.count(b"\n")
    return n

def read_head(fh_or_path, sep=",", n=3, **kw):
    return pd.read_csv(fh_or_path, sep=sep, nrows=n, dtype=str, keep_default_na=False, encoding_errors="replace", **kw)

def peek(rel: str, member: str | None = None, sep: str | None = None, n: int = 3):
    p = RAW / rel
    ext = ext_of(p)
    info = {"file": rel, "mb": round(p.stat().st_size / 1e6, 1)}
    head = None
    if ext in (".csv", ".txt"):
        s = sep or ("\t" if ext == ".txt" else ",")
        head = read_head(p, sep=s, n=n)
        info["lines"] = count_lines(p) - 1
    elif ext == ".csv.gz":
        head = read_head(p, n=n, compression="gzip")
    elif ext == ".zip":
        with zipfile.ZipFile(p) as z:
            names = [i for i in z.infolist() if not i.is_dir()]
            info["members"] = len(names)
            m = next(i for i in names if (member is None and i.filename.lower().endswith((".csv", ".txt", ".xlsx"))) or i.filename == member)
            info["member"] = f"{m.filename} ({m.file_size/1e6:.1f} MB unzipped)"
            with z.open(m) as fh:
                head = pd.read_excel(io.BytesIO(fh.read()), nrows=n, dtype=str) if m.filename.lower().endswith(".xlsx") \
                       else read_head(fh, sep=sep or ",", n=n)
    elif ext == ".xlsx":
        wb = load_workbook(p, read_only=True)
        info["sheets"] = {ws.title: f"{ws.max_row} x {ws.max_column}" for ws in wb.worksheets}
        wb.close()
    if head is not None:
        info["columns"] = head.shape[1]
    display(Markdown(f"**`{rel}`** · " + " · ".join(f"{k}: {v}" for k, v in info.items() if k != 'file')))
    if head is not None:
        display(head)
    return head

### 6a. Jobs and openings (demand)

In [10]:
peek("jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv");

**`jobs/nysdol_projections/pqm4-9qqb_2026-10-12.csv`** · mb: 0.7 · lines: 6758 · columns: 11

,Area,Period,SOC,Occupation Title,Base Year,Projected Year,Change,Percent,Total Annual Openings,Growth Annual Openings,Replacement Annual Openings
0,New York Statewide,2024 - 2034,00-0000,"Total, All Occupations",10709250,10897970,188720,1.76,1166222,18872,1147350
1,New York Statewide,2024 - 2034,11-0000,Management Occupations,791980,819020,27040,3.41,63334,2704,60630
2,New York Statewide,2024 - 2034,11-1000,Top Executives,233850,234490,640,0.27,17954,64,17890


In [11]:
peek("jobs/nysdol_projections_archive/pqm4-9qqb_2016-2026_wayback20210324183534.csv");

**`jobs/nysdol_projections_archive/pqm4-9qqb_2016-2026_wayback20210324183534.csv`** · mb: 0.7 · lines: 6626 · columns: 11

,Area,Period,SOC,Occupation Title,Base Year,Projected Year,Change,Percent,Total Annual Openings,Growth Annual Openings,Replacement Annual Openings
0,New York State,2018 - 2028,'00-0000,"Total, All Occupations",10489880,11577400,1087520,10.4,1345500,108750,1236750
1,New York State,2018 - 2028,'11-0000,Management Occupations,625980,698130,72150,11.5,62620,7220,55410
2,New York State,2018 - 2028,'11-1000,Top Executives,190130,206740,16610,8.7,18590,1660,16920


In [12]:
peek("jobs/bls_oews_metro/oesm25ma.zip", member="oesm25ma/MSA_M2025_dl.xlsx");

**`jobs/bls_oews_metro/oesm25ma.zip`** · mb: 39.9 · members: 3 · member: oesm25ma/MSA_M2025_dl.xlsx (30.6 MB unzipped) · columns: 32

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,TOT_EMP,EMP_PRSE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PCT_RPT,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,00-0000,All Occupations,total,75070,0,1000,1,NaN,NaN,26.13,54360,0.6,11.97,14.9,21.32,29.92,45.19,24900,30980,44340,62230,93990,NaN,NaN
1,10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-0000,Management Occupations,major,5470,2.1,72.874,1.02,NaN,NaN,53.94,112190,0.9,22.98,31.25,45.9,64.34,92.98,47790,64990,95480,133840,193390,NaN,NaN
2,10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,40,10,0.542,0.41,NaN,NaN,99.14,206200,6.3,42.5,58.65,79.7,113.51,187.68,88400,121980,165770,236100,390370,NaN,NaN


In [13]:
peek("jobs/nysdol_oews_regions/gkgz-nw24_2026-10-12.csv");

**`jobs/nysdol_oews_regions/gkgz-nw24_2026-10-12.csv`** · mb: 0.7 · lines: 7036 · columns: 10

,Area Type,Area,Area Name,Standard Occupational Code,Occupational Title,Employment,Mean Wage,Median Wage,Entry Wage,Experienced Wage
0,1,36,New York State,00-0000,"Total, All Occupations",9688000,84311,61012,39230,106852
1,1,36,New York State,11-0000,Management Occupations,676270,189747,168051,91597,238821
2,1,36,New York State,11-1011,Chief Executives,8430,357052,262311,156045,457557


In [14]:
peek("jobs/bls_ep_education/education_2026-10-12.xlsx");

**`jobs/bls_ep_education/education_2026-10-12.xlsx`** · mb: 0.1 · sheets: {'Index': '5 x 1', 'Table 5.1': '15 x 3', 'Table 5.2': '15 x 8', 'Table 5.3': '837 x 9', 'Table 5.4': '834 x 6'}

### 6b. Industries and hiring

In [15]:
peek("jobs/nysdol_qcew_annual/shc7-xcbw.csv");

**`jobs/nysdol_qcew_annual/shc7-xcbw.csv`** · mb: 128.0 · lines: 1163386 · columns: 10

,Area Type,Area,Ownership,NAICS,NAICS Title,Year,Establishments,Average Employment,Total Wage,Annual Average Salary
0,State,New York State,Private,11,"Agriculture, Forestry, Fishing and Hunting",2025,3401,29576,1458136146,49301
1,State,New York State,"Total, All Ownerships",11,"Agriculture, Forestry, Fishing and Hunting",2025,3401,29576,1458136146,49301
2,State,New York State,Private,111,Crop Production,2025,1508,13622,594867687,43670


In [16]:
peek("jobs/nysdol_ces/6k74-dgkb.csv");

**`jobs/nysdol_ces/6k74-dgkb.csv`** · mb: 37.1 · lines: 498400 · columns: 7

,Area,Area Name,Series,Title,Year,Month,Current Employment
0,15380,Buffalo-Cheektowaga Metro Area,90932622,Local Government Hospitals,1995,4,2600
1,15380,Buffalo-Cheektowaga Metro Area,90932622,Local Government Hospitals,1995,3,2600
2,15380,Buffalo-Cheektowaga Metro Area,90932622,Local Government Hospitals,1995,2,2600


In [17]:
peek("hiring/census_qwi/qwi_ny_sa_f_gc_ns_op_u.csv.gz");

**`hiring/census_qwi/qwi_ny_sa_f_gc_ns_op_u.csv.gz`** · mb: 167.2 · columns: 81

,periodicity,seasonadj,geo_level,geography,ind_level,industry,ownercode,sex,agegrp,race,ethnicity,education,firmage,firmsize,year,quarter,agg_level,Emp,EmpEnd,EmpS,...,sHirAEndR,sSepBegR,sHirAEndReplR,sHirAS,sHirNS,sSepS,sSepSnx,sTurnOvrS,sFrmJbGn,sFrmJbLs,sFrmJbC,sFrmJbGnS,sFrmJbLsS,sFrmJbCS,sEarnS,sEarnBeg,sEarnHirAS,sEarnHirNS,sEarnSepS,sPayroll
0,Q,U,C,36001,A,00,A05,0,A00,A0,A0,E0,0,0,2000,1,4129,,169056,,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,5
1,Q,U,C,36001,A,00,A05,0,A01,A0,A0,E0,0,0,2000,1,4131,,6559,,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,5
2,Q,U,C,36001,A,00,A05,0,A02,A0,A0,E0,0,0,2000,1,4131,,10750,,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,5


In [18]:
peek("jobs/bea_county_gdp/CAGDP2_2026-10-12.zip");

**`jobs/bea_county_gdp/CAGDP2_2026-10-12.zip`** · mb: 15.3 · members: 55 · member: CAGDP2__ALL_AREAS_2001_2024.csv (26.5 MB unzipped) · columns: 32

,GeoFIPS,GeoName,Region,TableName,LineCode,IndustryClassification,Description,Unit,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"""00000""",United States,,CAGDP2,1,...,All industry total,Thousands of dollars,10581929000,10929108000,11456450000,12217196000,13039197000,13815583000,14474228000,14769862000,14478067000,15048971000,15599732000,16253970000,16880683000,17608138000,18295019000,18804913000,19612102000,20656516000,21539982000,21375281000,23725645000,26054614000,27811517000,29298013000
1,"""00000""",United States,,CAGDP2,2,...,Private industries,Thousands of dollars,9188994000,9454709000,9904104000,10585649000,11327837000,12022487000,12585877000,12786770000,12431499000,12939469000,13461114000,14092708000,14665532000,15332504000,15951002000,16413054000,17156255000,18097765000,18909779000,18663991000,20917717000,23128892000,24712793000,26001672000
2,"""00000""",United States,,CAGDP2,3,11,"Agriculture, forestry, fishing and hunting",Thousands of dollars,100765000,96667000,115263000,144509000,130191000,127141000,146262000,146547000,129310000,145743000,179887000,179457000,215847000,200581000,182147000,167492000,176840000,177117000,164214000,164383000,230723000,294048000,270441000,269686000


In [19]:
peek("jobs/bls_ind_occ_matrix/matrix_2026-10-12.xlsx");

**`jobs/bls_ind_occ_matrix/matrix_2026-10-12.xlsx`** · mb: 7.9 · sheets: {'Matrix': '113899 x 14', 'Field Descriptions': '28 x 3'}

### 6c. Graduates (supply)

In [20]:
peek("graduates/ipeds_completions/C2024_A.zip");

**`graduates/ipeds_completions/C2024_A.zip`** · mb: 4.7 · members: 1 · member: C2024_a.csv (62.8 MB unzipped) · columns: 64

,UNITID,CIPCODE,MAJORNUM,AWLEVEL,XCTOTALT,CTOTALT,XCTOTALM,CTOTALM,XCTOTALW,CTOTALW,XCAIANT,CAIANT,XCAIANM,CAIANM,XCAIANW,CAIANW,XCASIAT,CASIAT,XCASIAM,CASIAM,...,XCWHITW,CWHITW,XC2MORT,C2MORT,XC2MORM,C2MORM,XC2MORW,C2MORW,XCUNKNT,CUNKNT,XCUNKNM,CUNKNM,XCUNKNW,CUNKNW,XCNRALT,CNRALT,XCNRALM,CNRALM,XCNRALW,CNRALW
0,100654,01.0999,1,5,R,12,R,2,R,10,Z,0,Z,0,Z,0,Z,0,Z,0,...,Z,0,R,1,Z,0,R,1,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0
1,100654,01.1001,1,5,R,10,R,1,R,9,Z,0,Z,0,Z,0,Z,0,Z,0,...,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0
2,100654,01.1001,1,7,R,7,R,1,R,6,Z,0,Z,0,Z,0,Z,0,Z,0,...,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0,Z,0,R,2,R,1,R,1


In [21]:
peek("graduates/ipeds_directory/HD2024.zip");

**`graduates/ipeds_directory/HD2024.zip`** · mb: 1.1 · members: 1 · member: HD2024.csv (4.5 MB unzipped) · columns: 72

,UNITID,INSTNM,IALIAS,ADDR,CITY,STABBR,ZIP,FIPS,OBEREG,CHFNM,CHFTITLE,GENTELE,EIN,UEIS,OPEID,OPEFLAG,WEBADDR,ADMINURL,FAIDURL,APPLURL,...,CARNEGIERSCH,CARNEGIESIZE,CARNEGIEALF,CARNEGIEAPM,CARNEGIEGPM,LANDGRNT,INSTSIZE,F1SYSTYP,F1SYSNAM,F1SYSCOD,CBSA,CBSATYPE,CSA,COUNTYCD,COUNTYNM,CNGDSTCD,LONGITUD,LATITUDE,DFRCGID,DFRCUSCG
0,100654,Alabama A & M University,AAMU,4900 Meridian Street,Normal,AL,35762,1,5,Dr. Daniel K. Wims,President,2563725000,636001097,JDVGS67MSLH7,00100200,1,www.aamu.edu/,https://www.aamu.edu/admissions-aid/index.html,https://www.aamu.edu/admissions-aid/financial-aid/,https://www.aamu.edu/admissions-aid/undergraduate-admiss...,...,3,3,4,1,2,1,3,2,-2 ...,-2,26620,1,290,1089,Madison County,105,-86.568502,34.783368,106,1
1,100663,University of Alabama at Birmingham,UAB,Administration Bldg Suite 1070,Birmingham,AL,35294-0110,1,5,Ray L. Watts,President,2059344011,636005396,YND4PLMC9AN7,00105200,1,https://www.uab.edu/,https://www.uab.edu/admissions/,https://www.uab.edu/cost-aid/,https://www.uab.edu/admissions/apply,...,1,4,5,2,8,2,5,1,The University of Alabama System ...,101050,13820,1,142,1073,Jefferson County,107,-86.799345,33.505697,91,1
2,100690,Amridge University,Southern Christian University Regions University,1200 Taylor Rd,Montgomery,AL,36117-3553,1,5,Michael C.Turner,President,33438738777550,237034324,RB27R4GLDKE7,02503400,1,https://www.amridgeuniversity.edu/,https://www.amridgeuniversity.edu/admissions/,https://www.amridgeuniversity.edu/financialaid/,https://www.amridgeuniversity.edu/myamridge/,...,-2,2,6,1,2,2,1,2,-2 ...,-2,33860,1,388,1101,Montgomery County,102,-86.174010,32.362609,125,2


In [22]:
peek("graduates/cuny_programs/28gk-bu58.csv");

**`graduates/cuny_programs/28gk-bu58.csv`** · mb: 16.7 · lines: 107676 · columns: 15

,IRP CODE,COLLEGE,PROGRAM NAME,AWARD NAME,AWARD NUMBER,DATE FIRST REGISTERED,DATE CLOSED,TAP ELIGIBLE,APTS ELIGIBLE,VVTA ELIGIBLE,CIP 2020 CODE,CIP 2020 TITLE SHORT,RECORD TYPE,PERIOD,COUNT
0,01912,Baruch College,Accountancy,MBA,1 of 1,01/01/1976,,YES,NO,YES,52.0301,Accounting.,Enrollment,Fall_2008,152
1,01912,Baruch College,Accountancy,MBA,1 of 1,01/01/1976,,YES,NO,YES,52.0301,Accounting.,Enrollment,Fall_2009,207
2,01912,Baruch College,Accountancy,MBA,1 of 1,01/01/1976,,YES,NO,YES,52.0301,Accounting.,Enrollment,Fall_2010,232


In [23]:
peek("graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx");

**`graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx`** · mb: 0.4 · sheets: {'File Guide': '8 x 2', 'CIP-SOC': '6098 x 4', 'SOC-CIP': '6094 x 4', 'New CIP': '1077 x 4', 'New SOC': '416 x 4', 'Added Matches': '1008 x 4', 'Unmatched CIP Codes': '195 x 4', 'Unmatched SOC Codes': '181 x 4'}

In [24]:
peek("graduates/nces_cip/Crosswalk2010to2020.csv");

**`graduates/nces_cip/Crosswalk2010to2020.csv`** · mb: 0.3 · lines: 2699 · columns: 6

,CIPCode2010,CIPTitle2010,Action,Text change,CIPCode2020,CIPTitle2020
0,"=""01""","AGRICULTURE, AGRICULTURE OPERATIONS, AND RELATED SCIENCES.",No substantive changes,yes,"=""01""",AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED...
1,"=""01.00""","Agriculture, General.",No substantive changes,no,"=""01.00""","Agriculture, General."
2,"=""01.0000""","Agriculture, General.",No substantive changes,no,"=""01.0000""","Agriculture, General."


### 6d. What grads actually do, skills, prices

In [25]:
peek("outcomes/acs_pums/csv_pny_2020-2024_5yr.zip");

**`outcomes/acs_pums/csv_pny_2020-2024_5yr.zip`** · mb: 140.1 · members: 2 · member: psam_p36.csv (638.8 MB unzipped) · columns: 285

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,STATE,ADJINC,PWGTP,AGEP,CIT,CITWP,COW,DDRS,DEAR,DEYE,DOUT,DPHY,DRAT,DRATX,...,PWGTP61,PWGTP62,PWGTP63,PWGTP64,PWGTP65,PWGTP66,PWGTP67,PWGTP68,PWGTP69,PWGTP70,PWGTP71,PWGTP72,PWGTP73,PWGTP74,PWGTP75,PWGTP76,PWGTP77,PWGTP78,PWGTP79,PWGTP80
0,P,2020GQ0000009,2,01,02001,1,36,1222017,1,77,1,,,1,2,2,1,1,,,...,2,1,1,1,0,2,0,1,1,0,1,1,1,1,1,1,2,1,0,0
1,P,2020GQ0000010,2,01,01500,1,36,1222017,16,22,1,,3,2,2,2,2,2,,,...,16,29,28,2,2,29,16,2,29,28,15,16,2,15,16,15,15,2,16,15
2,P,2020GQ0000013,2,01,01600,1,36,1222017,8,64,1,,,1,2,2,1,2,,,...,14,7,7,7,7,0,14,7,7,8,1,14,8,7,1,7,1,1,13,8


In [26]:
peek("skills/onet/db_31_0_csv.zip");

**`skills/onet/db_31_0_csv.zip`** · mb: 16.2 · members: 46 · member: db_31_0_csv/work_context.csv (52.9 MB unzipped) · columns: 16

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,4.C.1.a.2.c,Public Speaking,CX,Context,,3.07,37,0.2851,2.4923,3.6486,N,,08/2023,Incumbent
1,11-1011.00,Chief Executives,4.C.1.a.2.c,Public Speaking,CXP,Context (Categories 1-5),1,0.13,37,0.1370,0.0160,1.0770,N,,08/2023,Incumbent
2,11-1011.00,Chief Executives,4.C.1.a.2.c,Public Speaking,CXP,Context (Categories 1-5),2,39.49,37,11.0101,20.4073,62.4299,N,,08/2023,Incumbent


In [27]:
peek("bls_timeseries/bls_cu/cu-data-1-AllItems_2026-10-12.txt");

**`bls_timeseries/bls_cu/cu-data-1-AllItems_2026-10-12.txt`** · mb: 2.7 · lines: 64007 · columns: 5

,series_id,year,period,value,footnote_codes
0,CUSR0000SA0,1947,M01,21.48,
1,CUSR0000SA0,1947,M02,21.62,
2,CUSR0000SA0,1947,M03,22.00,


In [28]:
peek("jobs/bls_soc_crosswalk/soc_2010_to_2018_crosswalk.xlsx");

**`jobs/bls_soc_crosswalk/soc_2010_to_2018_crosswalk.xlsx`** · mb: 0.0 · sheets: {'Sorted by 2010': '909 x 4'}

### 6e. Traps visible on the first look

The previews above showed a few things that would silently break a naive load. Each one is checked below so it's on record, not just eyeballed.

In [29]:
# (1) Each NYS DOL projections file mixes two vintages: regions (incl. NYC) are one edition, statewide is 2 years newer.
#     Period text also changes format ("2020-2030" vs "2020 - 2030"), and the statewide area name changes.
proj = []
for f in sorted((RAW / "jobs").glob("nysdol_projections*/*.csv")):
    d = pd.read_csv(f, dtype=str, keep_default_na=False)
    d["Area"] = d["Area"].where(d["Area"] == "New York City", "statewide/other regions")
    proj.append(d.assign(file=f.name).groupby(["file", "Area"])["Period"].agg(lambda s: sorted(set(s))).to_frame())
display(pd.concat(proj).unstack("Area"))

# (2) The archived projections write SOC codes with a leading apostrophe (an Excel text marker)
arch = pd.read_csv(RAW / "jobs/nysdol_projections_archive/pqm4-9qqb_2016-2026_wayback20210324183534.csv", dtype=str, keep_default_na=False)
print("SOC codes starting with an apostrophe (2016-26 archive):", arch.SOC.str.startswith("'").sum(), "of", len(arch))

Period                            
Area                                           New York City     statewide/other regions
file                                                                                    
pqm4-9qqb_2016-2026_wayback20210324183534.csv  [2016 - 2026]  [2016 - 2026, 2018 - 2028]
pqm4-9qqb_2018-2028_wayback20221130141045.csv  [2018 - 2028]  [2018 - 2028, 2020 - 2030]
pqm4-9qqb_2020-2030_wayback20241104204940.csv    [2020-2030]    [2020-2030, 2022 - 2032]
pqm4-9qqb_2026-10-12.csv                       [2022 - 2032]  [2022 - 2032, 2024 - 2034]

SOC codes starting with an apostrophe (2016-26 archive): 6626 of 6626


In [30]:
# (3) NYS DOL wage table: column names carry leading spaces, and the last one is padded with ~200 spaces
w = pd.read_csv(RAW / "jobs/nysdol_oews_regions/gkgz-nw24_2026-10-12.csv", dtype=str, keep_default_na=False)
print([f"{c!r} (len {len(c)})" for c in w.columns])
print("Rows for NYC:", (w[" Area Name"].str.strip() == "New York City").sum())

# (4) NCES CIP 2010->2020 crosswalk stores codes as Excel formulas: ="01.0000"
cw = pd.read_csv(RAW / "graduates/nces_cip/Crosswalk2010to2020.csv", dtype=str, keep_default_na=False)
print("CIP codes wrapped as =\"...\":", cw.CIPCode2010.str.startswith('="').sum(), "of", len(cw))

# (5) Statewide files: QCEW and CES cover all of New York State; NYC is a subset we must filter to
q = pd.read_csv(RAW / "jobs/nysdol_qcew_annual/shc7-xcbw.csv", dtype=str, keep_default_na=False, usecols=["Area Type", "Area", "Year"])
print("\nQCEW rows by area type:"); print(q["Area Type"].value_counts().to_string())
print("QCEW years:", q.Year.min(), "->", q.Year.max(), "| NYC borough rows:",
      q.Area.isin(["Bronx County", "Kings County", "New York County", "Queens County", "Richmond County"]).sum())

["'Area Type' (len 9)", "' Area' (len 5)", "' Area Name' (len 10)", "' Standard Occupational Code' (len 27)", "' Occupational Title' (len 19)", "' Employment' (len 11)", "'Mean Wage' (len 9)", "' Median Wage' (len 12)", "' Entry Wage' (len 11)", "' Experienced Wage                                                                                                                                                                                                                                             ' (len 254)"]
Rows for NYC: 702
CIP codes wrapped as ="...": 2155 of 2699



QCEW rows by area type:
Area Type
County                           666122
Metropolitan Statistical Area    195640
Labor Market Area                182165
Workforce Investment Region       62613
Metropolitan Division             34475
State                             22371
QCEW years: 2000 -> 2025 | NYC borough rows: 73672


## 7. What we learned

**Formats.** 253 files, 2.82 GB. No JSON and no live API feed. Everything is a file:
- **CSV** is the main format: 30 plain CSVs, plus 218 CSVs inside zips (IPEDS, ACS PUMS, O*NET, BEA).
- **Excel** (17 files + 47 inside zips): OEWS, the BLS education/matrix tables, crosswalks.
- **Tab-separated `.txt`**: the BLS time-series flat files (CPI, JOLTS, LAUS).
- **Gzipped CSV**: Census QWI.
- **HTML**: 38 NYSED licensing pages (later) + 88 prototype BLS pages.

The Socrata portals *can* serve JSON, but full CSV exports are simpler and match what the portal publishes.

**How we get it (Extract).**
- **Bulk file download** is the backbone: 20 sources, 92 files, 1.9 GB.
- **Socrata open-data API**: 11 full exports (all the NYS DOL and CUNY tables) + 1 server-side query (NYC payroll, which deliberately skips employee names).
- **Wayback Machine**: the 3 older NYC projection editions NYS DOL overwrote.
- **Web pages to scrape**: NYSED licensing, all in "later".

**Provenance.** All 155 manifest records are on disk, and every byte count matches. The 98 files without a record are prototype leftovers:
- 6 of the 10 top-level files are byte-identical copies of registry downloads.
- 4 are unique: `HD2024_Dict.zip`, the IES crosswalk PDF, `Job Zones.txt`, and an older projections export.
- The 88 `bls_matrix/*.html` pages are replaced by `matrix.xlsx`.

**v1 cut.** 16 v1 + 5 support sources = 67 files, ~1.1 GB. Two sources on disk aren't in the brief cut at all:
- `bls_oews_research`: state × industry × job, 6 xlsx, ~846 MB.
- `bls_la`: LAUS unemployment.

What one row is (first look; confirmed per source in 02+):

| Source | One row = | Note |
|---|---|---|
| NYS DOL projections | area × edition × SOC | Summary rows (00-0000, 11-0000) mixed in with detailed jobs |
| OEWS metro | area × SOC (`O_GROUP` total→detailed) | All US metros; NY = `35620`. Blank cells become NaN even with `dtype=str` in `read_excel` |
| NYS DOL wages | area × SOC | 702 NYC rows; has the Entry Wage column (brief O4) |
| QCEW | area × ownership × NAICS × year | 1.16M lines, statewide; 73,672 borough rows. "Private" and "Total" ownership both present, a double-count risk |
| CES | area × series × month | Statewide; NYC area code 93561 |
| QWI | county × industry × age group × quarter | 81 columns; each measure has a status-flag twin (`s…`) |
| BEA GDP | area × industry line | **Wide**: one column per year 2001–2024, so it needs unpivoting |
| BLS staffing matrix | industry × job | 113,899 rows, national |
| IPEDS completions | college × CIP × major # × award level | `X…` columns are data-status flags next to each count |
| IPEDS directory | college | National; filter to NYC counties / CUNY system |
| CUNY Programs | program × record type × period | **Long**: enrollment and graduates stacked |
| ACS PUMS | person | 285 columns, 639 MB unzipped, 80 replicate weights. Load selected columns only |
| O*NET 31.0 | job × element × scale | 46 CSVs, long format |
| CPI | series × month | Includes US and every area; we need `CUURS12ASA0` |

Traps found (6e), each one silent in a naive load:
1. Every projections file mixes two editions: NYC/regions are one edition, and the statewide rows are 2 years newer. Filtering on "the file" is not filtering on the edition.
2. The period text changes format (`2020-2030` vs `2020 - 2030`), and the statewide name changes (`New York State` → `New York Statewide`).
3. The archived projections prefix every SOC code with `'`.
4. The NYS DOL wage headers have leading spaces, and one is padded to 254 characters.
5. NCES writes CIP codes as Excel formulas (`="01.0000"`, 2,155 of 2,699 rows).
6. QCEW, CES, the IPEDS directory and OEWS are statewide or national; NYC is a filter we must apply and log.

## 8. The ETL this project demonstrates

Strictly, the shape is ELT: land the raw files untouched, load them into the database, then transform with SQL. That's the modern pattern, and it's what the brief's `staging → core → mart` layout already describes.

| Stage | What happens | Where | Skill it shows |
|---|---|---|---|
| **Extract** | One registry drives every download: Socrata API, bulk files, Wayback. SHA-256 manifest; dated snapshots of sources that overwrite themselves | `src/sources.py`, notebook 00, `data/raw/` | API + bulk ingestion, idempotent downloads, data lineage |
| **Stage** | One typed table per source: trim headers, strip `'` / `="…"`, move suppression markers to flag columns, filter to NYC, drop summary rows | notebooks 02+ → PostgreSQL `staging` | Profiling, cleaning, documented decisions |
| **Core** | Shared dimensions: job (SOC 2018), major (CIP 2020), industry (NAICS), college, area, year, career cluster. Crosswalks SOC 2010→2018, CIP 2010→2020, CIP→SOC | PostgreSQL `core` | Dimensional modeling (star schema), crosswalks |
| **Mart** | One view per dashboard question | PostgreSQL `mart` | SQL: CTEs, window functions, views |
| **Validate** | Tie to published totals, unique grain, join fan-out checks | Every notebook + SQL tests | Data quality testing |
| **Serve** | Power BI on the database, published as an App | Power BI | BI modeling, DAX, data storytelling |

Still open (a Tier 2 decision, researched before we pick): what runs the Transform step.
- **A.** Notebooks + plain SQL files (current plan; simplest).
- **B.** Notebooks for staging, dbt-core for core/mart. dbt adds built-in tests, lineage docs and a widely recognized tool on the resume.
- **C.** Add an orchestrator (Prefect/Dagster/Airflow). Probably overkill for data that refreshes yearly.